# 🤖 Notebook 3 — Obligation Extractor
## What this notebook does
Sends each contract chunk to the Groq LLM API and extracts structured obligations.

**Simple explanation:**  
Imagine hiring a smart legal assistant who reads every paragraph  
and highlights: "This is a payment clause", "This is an SLA", "This has a deadline."

**Technical explanation:**  
Uses prompt engineering to instruct Llama-3-70B to extract obligations  
in JSON format. Each chunk is processed individually. The LLM returns  
structured data: obligation type, text, clause reference, deadline, risk level.


## Step 1 — Setup

In [ ]:
import os, json, re, time
from groq import Groq
from dotenv import load_dotenv
from sqlalchemy import create_engine, Column, Integer, String, Text, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker
from datetime import datetime
import spacy

load_dotenv()

# Groq client
api_key = os.getenv("GROQ_API_KEY")
if not api_key or "paste_your" in api_key:
    raise ValueError("❌ Set your GROQ_API_KEY in the .env file first!")

client = Groq(api_key=api_key)

# Database
engine  = create_engine("sqlite:///database/contractiq.db", echo=False)
Session = sessionmaker(bind=engine)
session = Session()

# spaCy for date extraction
try:
    nlp = spacy.load("en_core_web_sm")
    print("✅ spaCy loaded")
except:
    print("⚠️  spaCy model not found. Run: python -m spacy download en_core_web_sm")
    nlp = None

print(f"✅ Groq client ready | Key: {api_key[:8]}...{api_key[-4:]}")

## Step 2 — Define Obligation Types

**Simple:** Categories of promises found in contracts.

**Technical:** Taxonomy derived from the CUAD dataset (41 clause types) simplified into 9 categories relevant to enterprise IT contracts.

In [ ]:
# The 9 types of obligations we extract
OBLIGATION_TYPES = [
    "SLA",              # Service Level / Uptime commitments
    "Payment",          # Invoice, payment deadline, terms
    "Deadline",         # Time-bound actions required
    "Compliance",       # Regulatory / certification requirements
    "Liability",        # Caps, limits, indemnification
    "Renewal",          # Contract renewal / termination notice
    "DataProtection",   # GDPR, privacy, data handling
    "Confidentiality",  # NDA, non-disclosure obligations
    "Penalty",          # Fines, service credits, penalties
]

# Risk keywords — used to auto-assign risk level
HIGH_RISK_WORDS   = ["terminate", "penalty", "fine", "breach", "gdpr",
                      "liability", "legal", "regulatory", "immediate", "critical"]
MEDIUM_RISK_WORDS = ["notice", "review", "report", "audit", "renew", "deadline"]

def assign_risk(text):
    """Assign risk level based on keywords in the obligation text."""
    text_lower = text.lower()
    if any(word in text_lower for word in HIGH_RISK_WORDS):
        return "HIGH"
    elif any(word in text_lower for word in MEDIUM_RISK_WORDS):
        return "MEDIUM"
    return "LOW"

print("✅ Obligation taxonomy defined")
print(f"   {len(OBLIGATION_TYPES)} obligation types: {', '.join(OBLIGATION_TYPES)}")

## Step 3 — The LLM Prompt

**Simple:** The instructions we give the AI to read contracts like a lawyer.

**Technical:** Zero-shot prompt with JSON output schema. The prompt uses chain-of-thought structure: classify first, then extract. System prompt establishes legal analyst persona.

In [ ]:
SYSTEM_PROMPT = """You are a senior legal contract analyst with 20 years of experience
reviewing enterprise IT contracts. You extract contractual obligations with precision.
You always respond with valid JSON only — no markdown, no explanation outside the JSON."""

def build_extraction_prompt(chunk_text, doc_name):
    """
    Builds the prompt sent to Groq for obligation extraction.
    
    Simple:    We're giving the AI a reading task with very specific
               instructions on what to look for and how to report it.
    
    Technical: Structured prompt with output schema definition.
               Forces JSON output for reliable parsing.
               Includes document context to help with clause references.
    """
    return f"""Analyze this contract text from document: {doc_name}

TASK: Extract ALL contractual obligations from the text below.

For each obligation found, return a JSON object with these exact fields:
- "type": one of {OBLIGATION_TYPES}
- "text": the obligation in clear simple English (max 150 words)
- "clause_ref": section/clause number if visible, else "Not specified"  
- "deadline": any time constraint mentioned (e.g. "45 days", "72 hours", "Net-30", "annually") else "None"
- "risk_level": "HIGH", "MEDIUM", or "LOW" based on business impact

Return a JSON array like: [{{"type": "...", "text": "...", "clause_ref": "...", "deadline": "...", "risk_level": "..."}}]

If no obligations are found, return an empty array: []

CONTRACT TEXT:
{chunk_text}

JSON RESPONSE:"""

print("✅ Prompt template defined")
print("\nSample prompt structure:")
print("  System: Legal analyst persona")
print("  User:   Document name + chunk text + JSON schema")
print("  Output: JSON array of obligations")

## Step 4 — The Extraction Function

**Simple:** Sends text to Groq, gets back a list of obligations.

**Technical:** Calls Groq chat completions API with llama3-70b-8192. Parses JSON response with fallback regex cleaning. Handles API rate limits with exponential backoff.

In [ ]:
def extract_obligations_from_chunk(chunk_text, doc_name, chunk_index):
    """
    Sends one chunk to Groq and gets back extracted obligations.
    
    Simple:    Asks the AI 'what obligations are in this paragraph?'
               and gets back a structured list.
    
    Technical: POST to Groq /v1/chat/completions with llama3-70b-8192.
               Response parsed as JSON. Retries on rate limit (429).
               Returns list of obligation dicts or empty list on failure.
    """
    prompt = build_extraction_prompt(chunk_text, doc_name)
    
    for attempt in range(3):   # retry up to 3 times
        try:
            response = client.chat.completions.create(
                model       = "llama3-70b-8192",
                messages    = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens  = 1500,
                temperature = 0.1,   # low = more consistent, factual output
            )
            
            raw = response.choices[0].message.content.strip()
            
            # ── Clean and parse JSON ──────────────────────────
            # Sometimes LLM adds markdown fences — strip them
            raw = re.sub(r'^```json\s*', '', raw)
            raw = re.sub(r'^```\s*',     '', raw)
            raw = re.sub(r'\s*```$',     '', raw)
            raw = raw.strip()
            
            obligations = json.loads(raw)
            
            if not isinstance(obligations, list):
                obligations = [obligations]   # wrap if single object
            
            return obligations
            
        except json.JSONDecodeError as e:
            print(f"   ⚠️  JSON parse error on chunk {chunk_index}: {str(e)[:50]}")
            return []
            
        except Exception as e:
            error_str = str(e)
            if "rate_limit" in error_str.lower() or "429" in error_str:
                wait = (attempt + 1) * 10
                print(f"   ⏳ Rate limit hit. Waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"   ❌ API error: {error_str[:80]}")
                return []
    
    return []

print("✅ extract_obligations_from_chunk() defined")

## Step 5 — Process All Documents

This is the main extraction run. Reads all chunks from DB, sends to Groq, saves obligations back to DB.

⏳ This will take a few minutes depending on number of chunks.

In [ ]:
from sqlalchemy import text as sql_text

# Get DB models
Base = declarative_base()

class Obligation(Base):
    __tablename__  = "obligations"
    __table_args__ = {'extend_existing': True}
    id         = Column(Integer, primary_key=True)
    doc_id     = Column(Integer)
    doc_name   = Column(String(255))
    ob_type    = Column(String(100))
    text       = Column(Text)
    clause_ref = Column(String(100))
    deadline   = Column(String(200))
    risk_level = Column(String(20))
    created_at = Column(DateTime, default=datetime.now)

Base.metadata.create_all(engine)

# ── Load all chunks from DB ───────────────────────────────────
chunks = session.execute(sql_text(
    "SELECT id, doc_id, doc_name, chunk_text, chunk_index FROM chunks ORDER BY doc_id, chunk_index"
)).fetchall()

print(f"📋 Total chunks to process: {len(chunks)}")
print(f"   (Each chunk = 1 Groq API call)\n")

total_obligations = 0
processed_docs    = set()

for i, chunk in enumerate(chunks):
    chunk_id, doc_id, doc_name, chunk_text, chunk_index = chunk
    
    if doc_name not in processed_docs:
        print(f"\n📄 Processing: {doc_name}")
        processed_docs.add(doc_name)
    
    print(f"   Chunk {chunk_index+1}...", end=" ")
    
    # Check if already extracted
    existing = session.execute(sql_text(
        f"SELECT COUNT(*) FROM obligations WHERE doc_id={doc_id} AND clause_ref LIKE '%chunk{chunk_index}%'"
    )).scalar()
    
    # Extract obligations from this chunk
    obligations = extract_obligations_from_chunk(chunk_text, doc_name, chunk_index)
    
    # Save to database
    saved = 0
    for ob in obligations:
        try:
            new_ob = Obligation(
                doc_id     = doc_id,
                doc_name   = doc_name,
                ob_type    = ob.get("type", "General"),
                text       = ob.get("text", ""),
                clause_ref = ob.get("clause_ref", "Not specified"),
                deadline   = ob.get("deadline", "None"),
                risk_level = ob.get("risk_level", assign_risk(ob.get("text", ""))),
            )
            session.add(new_ob)
            saved += 1
        except Exception as e:
            pass
    
    session.commit()
    total_obligations += saved
    print(f"✅ {saved} obligations found")
    
    time.sleep(0.5)   # small pause to respect rate limits

print(f"\n{'='*50}")
print(f"✅ EXTRACTION COMPLETE")
print(f"   Total obligations extracted: {total_obligations}")
print(f"   Documents processed: {len(processed_docs)}")
print(f"{'='*50}")
print("\n▶ Run Notebook 4 next: Conflict Detector")

## Step 6 — Review Extracted Obligations

In [ ]:
import pandas as pd

obligations = session.execute(sql_text(
    "SELECT doc_name, ob_type, risk_level, clause_ref, deadline, substr(text,1,100) FROM obligations ORDER BY doc_name, ob_type"
)).fetchall()

df = pd.DataFrame(obligations, columns=["Document","Type","Risk","Clause","Deadline","Obligation (preview)"])

print(f"📊 EXTRACTED OBLIGATIONS — {len(df)} total\n")
print(df.to_string(index=False, max_colwidth=60))
print(f"\n📊 BREAKDOWN BY TYPE:")
print(df.groupby("Type").size().sort_values(ascending=False).to_string())
print(f"\n📊 BREAKDOWN BY RISK:")
print(df.groupby("Risk").size().to_string())